In [1]:
import pandas as pd
import numpy as np
import spacy
import unicodedata
from collections import Counter
import language_tool_python

nlp = spacy.load("en_core_web_md")

In [2]:
def safe_div(x, y):
    return x / y if y else np.nan


def normalizar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = unicodedata.normalize("NFC", texto)
    texto = texto.replace("\ufeff", "")
    texto = texto.replace("\u200b", "")
    texto = texto.replace("\xa0", " ")
    texto = " ".join(texto.split())

    return texto.strip()

In [3]:
# Extractor de rasgos tipo criterial features
MODALS = {
    "can", "could", "may", "might", "must",
    "shall", "should", "will", "would"
}

DISCOURSE_MARKERS = {
    "however", "therefore", "moreover", "although", "because",
    "while", "whereas", "firstly", "secondly", "finally",
    "in addition", "on the other hand", "for example"
}


def has_present_perfect(token):
    """
    Aproximación: have/has + participio pasado.
    """
    if token.lemma_ != "have" or token.pos_ not in {"AUX", "VERB"}:
        return False

    for child in token.children:
        if child.tag_ == "VBN":
            return True

    return False


def is_passive(token):
    """
    Aproximación: verbo con auxiliar pasivo o sujeto pasivo.
    """
    if token.dep_ in {"nsubjpass", "auxpass"}:
        return True

    if token.tag_ == "VBN":
        deps = {child.dep_ for child in token.children}
        if "auxpass" in deps or "nsubjpass" in deps:
            return True

    return False


def is_relative_clause(token):
    return token.dep_ == "relcl"


def is_conditional(doc):
    """
    Aproximación simple: presencia de if + would/could/might/should.
    """
    has_if = any(t.lower_ == "if" for t in doc)
    has_modal = any(t.lemma_ in {"would", "could", "might", "should"} for t in doc)
    return has_if and has_modal


def count_gerunds(doc):
    return sum(1 for t in doc if t.tag_ == "VBG")


def count_to_infinitives(doc):
    return sum(
        1 for t in doc
        if t.lower_ == "to" and t.i + 1 < len(doc) and doc[t.i + 1].pos_ == "VERB"
    )


def count_complex_noun_phrases(doc):
    """
    Aproximación: noun chunks con 3 o más tokens.
    """
    return sum(1 for chunk in doc.noun_chunks if len(chunk) >= 3)


def count_discourse_markers(text):
    text_low = text.lower()
    return sum(1 for marker in DISCOURSE_MARKERS if marker in text_low)

In [4]:
# Función principal
def extraer_criterial_features(doc):
    words = [t for t in doc if t.is_alpha]
    sents = list(doc.sents)

    n_words = len(words)
    n_sents = len(sents)

    pos_counts = Counter(t.pos_ for t in words)
    dep_counts = Counter(t.dep_ for t in words)

    n_modals = sum(1 for t in words if t.lemma_ in MODALS or t.tag_ == "MD")
    n_present_perfect = sum(1 for t in words if has_present_perfect(t))
    n_passives = sum(1 for t in words if is_passive(t))
    n_relative_clauses = sum(1 for t in words if is_relative_clause(t))
    n_conditionals = int(is_conditional(doc))
    n_gerunds = count_gerunds(doc)
    n_to_infinitives = count_to_infinitives(doc)
    n_complex_np = count_complex_noun_phrases(doc)
    n_discourse_markers = count_discourse_markers(doc.text)

    n_subordinate = (
        dep_counts["advcl"]
        + dep_counts["ccomp"]
        + dep_counts["xcomp"]
        + dep_counts["relcl"]
        + dep_counts["acl"]
    )

    n_lexical = (
        pos_counts["NOUN"]
        + pos_counts["PROPN"]
        + pos_counts["VERB"]
        + pos_counts["ADJ"]
        + pos_counts["ADV"]
    )

    return {
        "cf_modals": n_modals,
        "cf_present_perfect": n_present_perfect,
        "cf_passives": n_passives,
        "cf_relative_clauses": n_relative_clauses,
        "cf_conditionals": n_conditionals,
        "cf_gerunds": n_gerunds,
        "cf_to_infinitives": n_to_infinitives,
        "cf_complex_noun_phrases": n_complex_np,
        "cf_discourse_markers": n_discourse_markers,
        "cf_subordinate_clauses": n_subordinate,

        "cf_modal_ratio": safe_div(n_modals, n_words),
        "cf_present_perfect_ratio": safe_div(n_present_perfect, n_words),
        "cf_passive_ratio": safe_div(n_passives, n_words),
        "cf_relative_clause_ratio": safe_div(n_relative_clauses, n_sents),
        "cf_subordination_ratio": safe_div(n_subordinate, n_sents),
        "cf_complex_np_ratio": safe_div(n_complex_np, n_sents),
        "cf_lexical_density": safe_div(n_lexical, n_words),
    }

In [5]:
corpus = pd.read_csv("corpus_inmersia_0.csv")

In [6]:
corpus["text_norm"] = corpus["text_full"].apply(normalizar_texto)

docs = list(
    nlp.pipe(
        corpus["text_norm"],
        batch_size=50
    )
)

criterial_features = pd.DataFrame(
    [extraer_criterial_features(doc) for doc in docs]
)

corpus_criterial = pd.concat(
    [
        corpus.reset_index(drop=True),
        criterial_features.reset_index(drop=True)
    ],
    axis=1
)

In [7]:
corpus_criterial[
    [
        "doc_name",
        "cf_modals",
        "cf_present_perfect",
        "cf_passives",
        "cf_relative_clauses",
        "cf_conditionals",
        "cf_complex_noun_phrases",
        "cf_subordination_ratio",
        "cf_lexical_density"
    ]
].head()

,doc_name,cf_modals,cf_present_perfect,cf_passives,cf_relative_clauses,cf_conditionals,cf_complex_noun_phrases,cf_subordination_ratio,cf_lexical_density
0,GN_doc_1,2,0,0,3,1,6,1.000000,0.518987
1,GN_doc_2,0,0,0,3,0,7,0.846154,0.522581
2,GN_doc_3,1,0,0,4,0,7,1.636364,0.486034
3,GN_doc_4,2,0,3,6,0,3,3.400000,0.500000
4,GN_doc_5,3,0,2,1,1,6,0.357143,0.427586


In [8]:
corpus_criterial[corpus_criterial["cf_present_perfect"] > 0]

,doc_name,text_full,archivo_origen,text_norm,cf_modals,cf_present_perfect,cf_passives,cf_relative_clauses,cf_conditionals,cf_gerunds,...,cf_complex_noun_phrases,cf_discourse_markers,cf_subordinate_clauses,cf_modal_ratio,cf_present_perfect_ratio,cf_passive_ratio,cf_relative_clause_ratio,cf_subordination_ratio,cf_complex_np_ratio,cf_lexical_density
31,GN_doc_32,I am Alejandro and If you need informations ab...,73434020_data,I am Alejandro and If you need informations ab...,2,1,0,5,0,0,...,3,1,14,0.014184,0.007092,0.000000,0.833333,2.333333,0.500000,0.439716
55,GN_doc_56,"Hello readers, as all of you know im a person ...",000193-HOJA_data,"Hello readers, as all of you know im a person ...",3,1,0,5,0,2,...,12,2,18,0.013043,0.004348,0.000000,0.263158,0.947368,0.631579,0.430435
57,GN_doc_58,"When I have 10 years, my unforgatable celebrat...",000598-HOJA_data,"When I have 10 years, my unforgatable celebrat...",0,1,6,1,0,0,...,6,4,11,0.000000,0.005076,0.030457,0.083333,0.916667,0.500000,0.446701
148,GN_doc_149,Hello to all I am going to tell to you the bes...,07283982_data,Hello to all I am going to tell to you the bes...,0,1,0,7,0,1,...,3,2,17,0.000000,0.005464,0.000000,0.875000,2.125000,0.375000,0.420765
240,GN_doc_241,Today I haved got my cousins birthday celebrat...,21070822_data,Today I haved got my cousins birthday celebrat...,0,1,0,4,0,4,...,9,1,13,0.000000,0.006579,0.000000,0.666667,2.166667,1.500000,0.440789
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2250,CMA_doc_930,Hello everyone\n- I'm going to tell you about ...,78770071_data,Hello everyone - I'm going to tell you about a...,1,1,3,2,0,3,...,7,1,22,0.004975,0.004975,0.014925,0.250000,2.750000,0.875000,0.417910
2258,CMA_doc_938,In this blog I will writte for my expersence. ...,78773867_data,In this blog I will writte for my expersence. ...,2,1,5,1,0,0,...,5,0,9,0.014925,0.007463,0.037313,0.111111,1.000000,0.555556,0.462687
2289,CMA_doc_969,"Dear followers,\nIn this blog, I will talk abo...",78841490_data,"Dear followers, In this blog, I will talk abou...",2,1,1,2,0,0,...,4,3,13,0.012195,0.006098,0.006098,0.181818,1.181818,0.363636,0.439024
2384,CNAI_doc_70,Dear Ryan.\nI have seen your advert and I am i...,73122138_data,Dear Ryan. I have seen your advert and I am in...,1,1,0,0,0,0,...,3,1,10,0.009091,0.009091,0.000000,0.000000,1.250000,0.375000,0.445455


In [9]:
corpus_criterial.to_csv(
    "corpus_inmersia_cf.csv",
    index=False
)

In [10]:
tool = language_tool_python.LanguageTool("en-US")

In [11]:
def extraer_errores_languagetool(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return {
            "err_total": 0,
            "err_per_100_words": np.nan,
            "err_categories": {},
            "err_rules": {},
            "err_messages": []
        }

    texto = str(texto)
    matches = tool.check(texto)

    words = texto.split()
    n_words = len(words)

    categories = Counter()
    rules = Counter()
    messages = []

    for m in matches:
        rule_id = getattr(m, "rule_id", None)
        if rule_id is None:
            rule_id = getattr(m, "ruleId", "UNKNOWN_RULE")

        category = getattr(m, "category", "UNKNOWN_CATEGORY")

        categories[category] += 1
        rules[rule_id] += 1

        messages.append({
            "rule_id": rule_id,
            "category": category,
            "message": getattr(m, "message", ""),
            "context": getattr(m, "context", ""),
            "suggestions": getattr(m, "replacements", [])[:5]
        })

    return {
        "err_total": len(matches),
        "err_per_100_words": len(matches) / n_words * 100 if n_words else np.nan,
        "err_categories": dict(categories),
        "err_rules": dict(rules),
        "err_messages": messages
    }

In [12]:
errores = corpus_criterial["text_norm"].apply(extraer_errores_languagetool)

In [13]:
corpus_criterial["err_total"] = errores.apply(lambda x: x["err_total"])
corpus_criterial["err_per_100_words"] = errores.apply(lambda x: x["err_per_100_words"])
corpus_criterial["err_categories"] = errores.apply(lambda x: x["err_categories"])
corpus_criterial["err_rules"] = errores.apply(lambda x: x["err_rules"])
corpus_criterial["err_messages"] = errores.apply(lambda x: x["err_messages"])

In [14]:
all_categories = Counter()

for cats in corpus_criterial["err_categories"]:
    all_categories.update(cats)

all_categories

Counter({'TYPOS': 15835,
         'PUNCTUATION': 3749,
         'GRAMMAR': 3494,
         'CASING': 1555,
         'MISC': 635,
         'CONFUSED_WORDS': 597,
         'COLLOCATIONS': 483,
         'TYPOGRAPHY': 420,
         'REPETITIONS_STYLE': 324,
         'STYLE': 220,
         'REDUNDANCY': 198,
         'NONSTANDARD_PHRASES': 109,
         'COMPOUNDING': 36,
         'PROPER_NOUNS': 35,
         'MULTITOKEN_SPELLING': 34,
         'AMERICAN_ENGLISH_STYLE': 31,
         'SEMANTICS': 26,
         'BRITISH_ENGLISH': 13})

In [15]:
for cat in all_categories:
    corpus_criterial[f"err_cat_{cat}"] = corpus_criterial["err_categories"].apply(
        lambda x: x.get(cat, 0)
    )

In [16]:
doc_id = 0

corpus_criterial.loc[doc_id, "doc_name"], corpus_criterial.loc[doc_id, "err_messages"]

('GN_doc_1',
 [{'rule_id': 'MORFOLOGIK_RULE_EN_US',
   'category': 'TYPOS',
   'message': 'Possible spelling mistake found.',
   'context': 'A WONDERFUL PLACE Last year. I went to Pirynes with my family. We went in August and w...',
   'suggestions': ['Pirates', 'Pines', 'Prunes', 'Piranesi', 'Pinenes']},
  {'rule_id': 'COMMA_COMPOUND_SENTENCE',
   'category': 'PUNCTUATION',
   'message': 'Use a comma before ‘and’ if it connects two independent clauses (unless they are closely connected and short).',
   'context': '...irynes with my family. We went in August and we stayed in a hotel. I really love the...',
   'suggestions': [', and']},
  {'rule_id': 'MORFOLOGIK_RULE_EN_US',
   'category': 'TYPOS',
   'message': 'Possible spelling mistake found.',
   'context': '...we stayed in a hotel. I really love the pleace because it have got a very highs mounta...',
   'suggestions': ['place', 'please', 'peace', 'Peace', 'pleach']},
  {'rule_id': 'IT_VBZ',
   'category': 'GRAMMAR',
   'message': '